## Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import DataFrame

In [0]:
df_orders = spark.read.table("data_engineering_2026.bronze.orders")

df_customers = spark.read.table("data_engineering_2026.bronze.customers")

df_items = spark.read.table("data_engineering_2026.bronze.items")

df_crm = spark.read.table("data_engineering_2026.bronze.crm_cust_info")

df_crm_sales = spark.read.table("data_engineering_2026.bronze.crm_sales_details")

## Check 1 Null Values


    

In [0]:
def check_nulls(df, table_name):
    
    total = df.count()

    null_df = df.select([
        sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns])

    stack_expr = "stack({0}, {1}) as (column_name, null_count)".format(
        len(df.columns),
        ", ".join([f"'{c}', {c}" for c in df.columns]))

    output_df = null_df.selectExpr(stack_expr)\
        .withColumn("table_name", lit(table_name))\
        .withColumn("total_rows", lit(total))\
        .withColumn("issue_pct", round((col("null_count") / col("total_rows")) * 100, 2))\
        .withColumn("check_type", lit("NULL_CHECK"))

    return output_df.select(
        "table_name",
        "column_name",
        "check_type",
        col("null_count").alias("issue_count"),
        "total_rows",
        "issue_pct")

In [0]:
df_null_results = (
    check_nulls(df_orders, "orders")
    .union(check_nulls(df_customers, "customers"))
    .union(check_nulls(df_items, "items"))
    .union(check_nulls(df_crm, "crm_cust_info"))
    )

In [0]:
df_null_results.display()

In [0]:
df_null_results\
    .filter(col("issue_count") > 0)\
    .show(truncate=False)

## Duplicate Check

In [0]:
def check_duplicates(df, table_name, key_column):
    
    total = df.count()

    dup_df = df.groupBy(key_column)\
        .count()\
        .filter(col("count") > 1)

    dup_count_df = dup_df.select(
        sum(col("count") - 1).alias("duplicate_count"))

    dup_output = dup_count_df \
        .withColumn("table_name", lit(table_name))\
        .withColumn("column_name", lit(key_column))\
        .withColumn("total_rows", lit(total))\
        .withColumn("issue_pct", round((col("duplicate_count") / col("total_rows")) * 100, 2))\
        .withColumn("check_type", lit("DUPLICATE_CHECK"))

    return dup_output.select(
        "table_name",
        "column_name",
        "check_type",
        col("duplicate_count").alias("issue_count"),
        "total_rows",
        "issue_pct")

In [0]:
df_dup_results = (
    check_duplicates(df_orders, "orders", "order_id")
    .union(check_duplicates(df_customers, "customers", "customer_id"))
    .union(check_duplicates(df_items, "items", "order_id"))
    .union(check_duplicates(df_crm, "crm_cust_info", "cst_id"))
    )

df_dup_results.display()

## Future Data check

In [0]:
def future_dates(df, table_name, date_columns):

    if isinstance(date_columns, str):
        date_columns = [date_columns]

    total = df.count()

    future_df = df.select([
        sum(when(col(c).cast("timestamp") > current_timestamp(), 1)
            .otherwise(0)).alias(f"{c}_future")
        for c in date_columns])
    
    stack_expr = "stack({0}, {1}) as (column_name, future_count)".format(
        len(date_columns),
        ", ".join([f"'{c}', {c}_future" for c in date_columns]))

    result_df = future_df.selectExpr(stack_expr) \
        .withColumn("table_name", lit(table_name)) \
        .withColumn("total_rows", lit(total)) \
        .withColumn("issue_pct",
            when(lit(total) > 0, (col("future_count") / total) * 100)\
                .otherwise(0))\
        .withColumn("check_type", lit("FUTURE_DATE"))
        
    return result_df.select(
        "table_name",
        "column_name",
        "check_type",
        "future_count",
        "total_rows",
        "issue_pct")

In [0]:
df_orders.printSchema()


In [0]:
df_future_results = future_dates(df_orders,"orders",["order_pruchase_timestamp","order_approved_at","order_delivered_carrier_date","order_order_delivered_customer_date","order_estimated_delivery_date"])\
.union(future_dates(df_crm,"crm_cust_infor",["cst_create_date"]))

df_future_results.display()

In [0]:
df_future_results\
    .filter(col("future_count") > 0)\
    .display()

## Negative Value

In [0]:
def check_negative_simple(df, table_name, columns):
    
    total = df.count()
    
    result_df = df.select([
        sum(when(col(c) < 0, 1).otherwise(0)).alias(c)
        for c in columns])

    stack_expr = "stack({0}, {1}) as (column_name, negative_count)".format(
        len(columns),
        ", ".join([f"'{c}', {c}" for c in columns]))
    
    result_df = result_df.selectExpr(stack_expr) \
        .withColumn("table_name", lit(table_name)) \
        .withColumn("total_rows", lit(total))\
        .withColumn("negative_pct", (col("negative_count") / total) * 100)\
        .withColumn("check_type", lit("NEGATIVE_VALUE"))
        
    return result_df.select(
        "table_name",
        "column_name",
        "check_type",
        "negative_count",
        "total_rows",
        "negative_pct")

In [0]:
df_neg_results = check_negative_simple(df_items,"items",["price", "freight_value"]).union(
check_negative_simple(df_crm_sales,"crm_sales_details",["sls_price",]))

df_neg_results.display()

## Invalid_Status

In [0]:
def invalid_status(df, table_name, column, valid_values):
    
    total = df.count()

    output_df = df.select(
        sum(when(~col(column).isin(valid_values), 1).otherwise(0)).alias("invalid_count"))
    
    output_df = output_df \
        .withColumn("table_name", lit(table_name)) \
        .withColumn("column_name", lit(column)) \
        .withColumn("total_rows", lit(total)) \
        .withColumn("issue_pct", col("invalid_count") / total * 100) \
        .withColumn("check_type", lit("INVALID_STATUS"))

    return output_df.select(
        "table_name",
        "column_name",
        "check_type",
        col("invalid_count").alias("issue_count"),
        "total_rows",
        "issue_pct")

In [0]:
valid_statuses = [
    "delivered","shipped","canceled",
    "unavailable","invoiced","processing",
    "created","approved"]

df_status_results = invalid_status(df_orders,"orders","order_status",valid_statuses)

df_status_results.show()

## Referencial Integrity check

In [0]:
def check_referential(df_left, df_right, table_name, key_column):
    
    total = df_left.count()

    orphan_count = df_left.join(
        broadcast(df_right.select(key_column)),
        key_column,
        "left_anti"
    ).count()

    orphan_pct = (orphan_count / total) * 100 if total > 0 else 0

    result_df = spark.createDataFrame([(
        table_name,
        key_column,
        "REFERENTIAL_INTEGRITY",
        orphan_count,
        total,
        orphan_pct
    )], [
        "table_name",
        "column_name",
        "check_type",
        "issue_count",
        "total_rows",
        "issue_pct"
    ])

    return result_df

In [0]:
df_ref_results = check_referential(df_orders,df_customers,"orders","customer_id")

df_ref_results.show()

In [0]:
df_quality_results = (
    df_null_results
    .union(df_dup_results)
    .union(df_future_results)
    .union(df_neg_results)
    .union(df_status_results)
    .union(df_ref_results)
)

# Add run date
df_quality_results = df_quality_results.withColumn("run_date", current_date())

In [0]:
df_quality_results\
    .display()

## Writing the Table

In [0]:
df_quality_results.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_engineering_2026.silver.quality_result")